In [1]:
import torch
import torch.nn as nn

In [2]:
def conv_block(inp_filt, out_filt):
    conv = nn.Sequential(nn.Conv2d(inp_filt, out_filt, 3, padding=1),
                         nn.BatchNorm2d(out_filt),
                         nn.ReLU(inplace=True)
                         )
    return conv

In [3]:
class Encoder_Block(nn.Module):
    def __init__(self, inp_filt, out_filt):
        super().__init__()
        self.conv2k = nn.Sequential(
            conv_block(inp_filt, out_filt),
            conv_block(out_filt, out_filt)
        )
        self.x_2 = nn.MaxPool2d(2,2)

    def forward(self, x):
        skip = self.conv2k(x)
        out = self.x_2(skip)
        return skip, out

In [4]:
class Bro_UNET(nn.Module):
    def __init__(self, ifilt=3, nf=64):
        super().__init__()
        up_f = [128, 256, 512, 1024]
        d_f = [512, 256, 128, 64]
        self.enc_blk_1 = Encoder_Block(ifilt, nf) #x, x/2 .. x is skip, x/2 in input to next encoder block
        self.enc_blk_2 = Encoder_Block(nf, up_f[0]) #x/2, x/4
        self.enc_blk_3 = Encoder_Block(up_f[0], up_f[1]) #x/4, x/8
        self.enc_blk_4 = Encoder_Block(up_f[1], up_f[2]) #x/8, x/16

        self.bneck = conv_block(up_f[2], up_f[3]) #x/16
        self.bneck_x_2x = nn.ConvTranspose2d(up_f[3], d_f[0], (2,2), 2)

        self.dec_blk_enc_blk_3 = self.decoder_blk(d_f[0]) #512
        self.dec_blk_enc_blk_2 = self.decoder_blk(d_f[1]) #256
        self.dec_blk_enc_blk_1 = self.decoder_blk(d_f[2]) #128

        self.final_conv = nn.Sequential(
            conv_block(d_f[2], d_f[3]),
            nn.Conv2d(d_f[3], 1, 1),
            nn.Sigmoid()
            )

    def decoder_blk(self, inp_filt):
        conv_x_2x = nn.Sequential(
            conv_block(inp_filt*2, inp_filt),
            conv_block(inp_filt, inp_filt),
            nn.ConvTranspose2d(inp_filt, int(inp_filt/2), (2,2), 2)
            )
        return conv_x_2x

    def forward(self, x):
        s1, out = self.enc_blk_1(x)  #224, 112
        s2, out = self.enc_blk_2(out) #112, 56
        s3, out = self.enc_blk_3(out) #56, 28
        s4, out = self.enc_blk_4(out) #28, 14

        out = self.bneck(out)
        out = self.bneck_x_2x(out) #28

        out = torch.cat((s4, out), 1)
        out = self.dec_blk_enc_blk_3(out)
        out = torch.cat((s3, out), 1)
        out = self.dec_blk_enc_blk_2(out)
        out = torch.cat((s2, out), 1)
        out = self.dec_blk_enc_blk_1(out)
        out = torch.cat((s1, out), 1)
        out = self.final_conv(out)
        return out

In [5]:
inp = torch.ones(2,3,224, 224)
inp.shape

torch.Size([2, 3, 224, 224])

In [6]:
model = Bro_UNET()

In [7]:
!pip install torchinfo
from torchinfo import summary

In [8]:
out = model(inp)

In [9]:
out.shape

torch.Size([2, 1, 224, 224])

In [10]:
summary(model,input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
Bro_UNET                                 [1, 1, 224, 224]          --
├─Encoder_Block: 1-1                     [1, 64, 224, 224]         --
│    └─Sequential: 2-1                   [1, 64, 224, 224]         --
│    │    └─Sequential: 3-1              [1, 64, 224, 224]         1,920
│    │    └─Sequential: 3-2              [1, 64, 224, 224]         37,056
│    └─MaxPool2d: 2-2                    [1, 64, 112, 112]         --
├─Encoder_Block: 1-2                     [1, 128, 112, 112]        --
│    └─Sequential: 2-3                   [1, 128, 112, 112]        --
│    │    └─Sequential: 3-3              [1, 128, 112, 112]        74,112
│    │    └─Sequential: 3-4              [1, 128, 112, 112]        147,840
│    └─MaxPool2d: 2-4                    [1, 128, 56, 56]          --
├─Encoder_Block: 1-3                     [1, 256, 56, 56]          --
│    └─Sequential: 2-5                   [1, 256, 56, 56]          --

In [ ]:
def channel_attention_module(x, ratio=8):
    channel = x.shape[-1]
    
    l1 = Dense(channel//ratio, activation="relu", use_bias=False)
    l2 = Dense(channel, use_bias=False)
    
    x1 = GlobalAveragePooling2D()(x)
    x1 = l1(x1)
    x1 = l2(x1)
    
    x2 = GlobalMaxPooling2D()(x)
    x2 = l1(x2)
    x2 = l2(x2)
    
    feats = x1 + x2
    feats = Activation("sigmoid")(feats)
    
    feats = Multiply()([x, feats])
    return feats

In [ ]:
def channel_attention_module(x, ratio=8):
    channel = x.shape[-1]
    l1 = nn.Sequential(
        nn.Linear(channel, channel//ratio),
        nn.ReLU(inplace=True),
        nn.Linear(channel//ratio, channel, use_bias=False)
    )
    x1 = nn.AdaptiveAvgPool2d()    #GlobalAveragePooling2D()(x)
    x1 = l1(x1)
    x2 = nn.MaxPool2d()    #GlobalMaxPooling2D()(x)
    x2 = l1(x2)
    feats = x1 + x2
    feats = nn.Sigmoid(feats)\
    feats = x * feats
    return feats

In [ ]:
def spatial_attention_module(x):
    x1 = torch.mean(x, axis=-1)
    x1 = torch.unsqueeze(x1, dim=-1)
    
    x2 = torch.max(x, dim=-1)
    x2 = torch.unsqueeze(x2, dim=-1)
    
    feats = torch.cat((x1, x2), dim=-1)
    feats = nn.Conv2d(1, kernel_size=7, padding="same", activation="sigmoid")(feats)
    
    feats = Multiply()([x, feats])
    return feats